This notebook introduces `flow`, the SysML v2 construct for declaring item flows between parts; after running it you can model the material or signal interfaces in a structural decomposition and render an interconnection diagram.

`allocate` (Ch5 nb02) shows which part performs a function. `flow` shows what passes between parts at runtime. A `flow X.port to Y.port` statement creates a `FlowUsage` element connecting two `PartUsage` members by their item ports.

This notebook adds a `BreadHandling` assembly with a `BreadLoader` and `BreadEjector`, connected by the bread item flow. It then uses `build_interconnection_intent()` and `render_sysmld()` to produce an interconnection SVG.

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch05-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch05-cumulative.sysml` file adds two architectural constructs: `allocate ApplyHeat to HeatingSystem` records the functional-to-physical assignment, and a `BreadHandling` subsystem with `flow loader.bread to ejector.bread` expresses the item flow at the port level. These connect the functional layer (actions) to the structural layer (parts).

In [ ]:
# Negative control: a flow referencing a part usage that does not exist in the assembly
# raises "unresolved reference" for the undefined dotted path.
bad_source = """
package BadFlow {
    private import ScalarValues::*;
    item def Bread;
    part def Loader { part loaf : Bread; }
    part def Assembly {
        part loader : Loader;
        flow loader.loaf to undefined_ejector.loaf;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print(f"Neg control diagnostics: {bad.diagnostics[0].message!r}")

In [ ]:
from toaster.render import build_interconnection_intent, render_sysmld
from pathlib import Path
import tempfile, os

# Build the interconnection intent for BreadHandling
intent = build_interconnection_intent(model, "ToasterDemo::BreadHandling")
print(f"Parts: {[p['name'] for p in intent['parts']]}")
print(f"Flows: {intent['flows']}")

# Render to SVG
out_path = Path(tempfile.mkdtemp()) / "bread_handling.svg"
render_sysmld(intent, out_path)
print(f"SVG written: {out_path} ({os.path.getsize(out_path)} bytes)")

The `flow` relationship in SysML v2 (A-F) is parsed and stored in OpenSysML's element graph (O-S); `build_interconnection_intent()` extracts the endpoint paths via `sysx:sourceText` and `render_sysmld()` produces an SVG showing the `loader` → `ejector` item flow (E).

Try the chapter exercise in `exercises/ch05/exercise.ipynb`: add a `CoffeeFlow` part with a `pump` and a `filter`, declare a `flow pump.water to filter.water`, build the interconnection intent, and confirm the flow endpoint paths appear correctly.